# Train the humanizer moat on a free GPU (Colab/Kaggle)
Set the runtime to a **T4 GPU**. Private repo: put a GitHub token in the clone URL. Run top to bottom.
Always run the smoke cell first.

In [ ]:
!nvidia-smi
# private repo: GITHUB_TOKEN below, or make the repo public for the clone
GITHUB_TOKEN = ""  # @param {type:"string"}
url = f"https://{GITHUB_TOKEN}@github.com/ssamba1/untell.git" if GITHUB_TOKEN else "https://github.com/ssamba1/untell.git"
!git clone $url untell
%cd untell
!pip install -q -e ".[train,full]"

In [ ]:
# Smoke test FIRST (tiny model, 2 steps) — proves the pipeline runs before the real train.
!python -m untell.training.rl_humanizer --smoke

In [ ]:
# Real GRPO train. 3B fits a 16GB T4 with --load-4bit (QLoRA). Use Qwen2.5-1.5B for full LoRA.
!python -m untell.training.rl_humanizer --model Qwen/Qwen2.5-3B-Instruct --tier full --steps 500 --load-4bit
# DPO alternative (often steadier):
# !python -m untell.training.dpo_humanizer --model Qwen/Qwen2.5-3B-Instruct --tier full --load-4bit

In [ ]:
# Save the adapter so it survives disconnect (Colab). On Kaggle it's in the notebook Output.
from google.colab import drive; drive.mount('/content/drive')
!cp -r out/rl-humanizer /content/drive/MyDrive/